# T3 · Who gets the blame

**Tiny Transformer, Unit 2, module T3.** Every training step ends with one number, the loss, and 9,417 learned numbers to
move. How does the loss tell each one which way to move, and how strongly? Here you remake the run's first batch at step 0 from
its seed and find its gradients four ways: one bias's by its rule, by nudging, part by part back through the block, and the token
table's, one job at a time.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/T3-training-gradients.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook T3-training-gradients.ipynb`.

**Computed here:** the first batch at step 0 and its gradients, by backpropagation and by nudging. **Checked against the run:** its
recorded first batch and first loss, and the course's numbers, which the notebook asserts.

Laid out like the page: two sections, four chapters, and under each chapter its page's three steps (Picture it, Watch it in our
run, Read and run the code), then the code to run and a "your turn".

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import hashlib, os, urllib.request
import torch

torch.set_num_threads(1)     # a model this small runs fastest on one CPU thread
print(torch.__version__)

2.10.0+cpu


**The text, the model and this module's code.** Tiny Shakespeare, pinned by its sha256; then the model's file, `model.py`, copied
in whole; then Unit 2's teaching functions, `code/training_steps.py`, copied in whole: T3's are the last five.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
#@title Unit 2's teaching functions: code/training_steps.py, copied whole (run me) { display-mode: "form" }
"""Unit 2 · Inside the learning loop: the recorded run's training, station by station, done explicitly.

The teaching functions of Unit 2 (each T notebook carries this file whole, and the chapter pages light its lines; the first
line is a Colab form title, so the notebooks fold this cell and no chapter's question is answered by its code early). They sit
beside M6's code/training_loop.py, with its names (get_batch, loss_of, train_step, every_window, average_loss), and repeat the
recorded run's own code, training/prompter/train.py: test_training_steps.py checks them against it and against the run's
numbers (training/prompter/unit2.py's export, prompter-unit2.json).
"""
import math
import os

import torch


# ---------------------------------------------------------------------------------------------------- T1 · picking the batches

def check_starts(n_train, n_val, batches=100, B=64, T=32, seed=2024):
    """The checks' fixed windows, as train.py draws them: 100 batches of starts in the training text, then 100 in the held-back
    text."""
    g = torch.Generator().manual_seed(seed)                       # the checks' generator
    train = torch.stack([torch.randint(n_train - T - 1, (B,), generator=g) for _ in range(batches)])   # [batches×B]
    val = torch.stack([torch.randint(n_val - T - 1, (B,), generator=g) for _ in range(batches)])       # [batches×B]
    return train, val


def run_starts(n, steps=100_000, B=64, T=32, seed=1337, make_model=None):
    """Every step's B window starts, as the recorded run drew them (train.py): the main random stream, started by the seed,
    first fills the model's random starting numbers (make_model builds the model), then draws B starts a step."""
    torch.manual_seed(seed)                                       # the main stream: starting numbers, then every batch
    if make_model is not None:
        make_model()                                              # first: the model's random starting numbers
    return torch.stack([torch.randint(n - T - 1, (B,)) for _ in range(steps)])   # [steps×B]  then B starts every step


def times_drawn(starts, n, T=32):
    """How many times each possible start was drawn. randint(n - T - 1) never draws its limit: starts run 0 … n - T - 2."""
    return torch.bincount(starts.flatten(), minlength=n - T - 1)  # [n−T−1]   one count per possible start


def times_asked(starts, n, T=32):
    """How many times each character of the text was a right answer: a window from start i asks about i + 1 … i + T."""
    drawn = times_drawn(starts, n, T)                             # [n−T−1]
    change = torch.zeros(n + 1, dtype=torch.long)
    change[1:n - T] += drawn                                      # a window from i starts asking at i + 1 …
    change[T + 1:n] -= drawn                                      # … and stops after i + T
    return change.cumsum(0)[:n]                                   # [n]       one count per character


def times_through(n, steps=100_000, B=64, T=32):
    """How many times over the run reads its text: every step asks B × T questions, one per right answer."""
    return steps * B * T / n                                      # answers ÷ characters


# ---------------------------------------------------------------------------------------------------- T2 · the loss

def all_at_once(model, x, y):
    """A batch's right answers all at once: their probabilities multiplied, in the model's 32-bit numbers, and their costs,
    minus each one's log, added and averaged (the loss)."""
    probs = torch.softmax(model(x)[0], dim=-1)                    # [B×T×65]  probabilities, as M6's loss_of makes them
    p = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)             # [B×T]     each right answer's probability
    product = p.prod()                                            # []        all of them together: too small for 32 bits, 0.0
    total = -p.log().sum()                                        # []        their costs, added
    return product, total, total / p.numel()                      # the product, the sum, and the average: the loss


def loss_from_scores(model, x, y):
    """The loss as model.py's F.cross_entropy works it out, straight from the scores: for each answer, the log of the total of
    e to every score, minus the right answer's own score. No probability is ever made."""
    scores = model(x)[0]                                          # [B×T×65]  a score for every character
    right = scores.gather(-1, y.unsqueeze(-1)).squeeze(-1)        # [B×T]     each right answer's score
    return (scores.logsumexp(-1) - right).mean()                  # []        minus the log of each probability, averaged


def answer_costs(model, ids, T=32, per=256):
    """Every right answer's probability over a whole text, and whether it was the top pick, read as the final score reads it
    (train.py full_loss): windows side by side, every character scored once (but the last few, which fill no window)."""
    n = len(range(0, len(ids) - T - 1, T))                        # how many whole windows fit
    x, y = ids[:n * T].view(n, T), ids[1:n * T + 1].view(n, T)    # [n×T] each: the inputs, and the targets moved one along
    p, top = [], []
    with torch.no_grad():
        for xb, yb in zip(x.split(per), y.split(per)):
            probs = torch.softmax(model(xb)[0], dim=-1)           # [256×T×65]
            p.append(probs.gather(-1, yb.unsqueeze(-1)).squeeze(-1))   # [256×T]  the right answer's probability
            top.append(probs.argmax(-1) == yb)                    # [256×T]  was it the top pick?
    return torch.cat(p).flatten(), torch.cat(top).flatten()       # [n·T] each


def share_of_loss(p, below=0.01):
    """The answers given a probability under `below`: their share of all the answers, and their share of the loss."""
    cost = -p.log()                                               # each answer's cost
    rare = p < below
    return rare.float().mean().item(), (cost[rare].sum() / cost.sum()).item()


def sure_or_right(p, top):
    """Accuracy, the share of answers whose top pick is right; and the share right but unsure: top pick, yet under one half."""
    return top.float().mean().item(), (top & (p < 0.5)).float().mean().item()


def choices(loss):
    """A loss read backwards: an even guess among n characters costs ln n on every answer, so n = e to the loss."""
    return math.exp(loss)


# ---------------------------------------------------------------------------------------------------- T3 · the way back

def bias_gradient(probs, y):
    """Every readout bias's gradient, by its rule. A bias adds to one character's score at every place, so its gradient is that
    character's probability averaged over the places, minus its share of the right answers."""
    V = probs.size(-1)
    predicted = probs.reshape(-1, V).mean(0)                      # [65]  each character's average probability
    right = torch.bincount(y.flatten(), minlength=V).float() / y.numel()   # [65]  its share of the right answers
    return predicted - right                                      # [65]  average probability, minus share of right answers


def nudge(model, x, y, param, i, h=1e-3):
    """One number's gradient without backpropagation: nudge it up by h and run the batch, then down by h and run it again, and
    take the slope through the two losses. The number is put back as it was."""
    flat = param.data.view(-1)
    old = flat[i].item()
    with torch.no_grad():
        flat[i] = old + h
        up = model(x, y)[1].item()                                # the loss, nudged up
        flat[i] = old - h
        down = model(x, y)[1].item()                              # the loss, nudged down
        flat[i] = old                                             # put back
    return (up - down) / (2 * h)


def grad_twice(model, x, y, param):
    """backward fills every learned number's .grad in one pass, and moves none of them. Run it again without clearing, and the
    gradients add up: why train.py clears them first, with opt.zero_grad()."""
    model.zero_grad()
    model(x, y)[1].backward()                                     # every .grad filled; no number moved
    once = param.grad.clone()
    model(x, y)[1].backward()                                     # again, without clearing
    return once, param.grad.clone()                               # the second: twice the first


def backward_sizes(model, x, y):
    """The gradient on its way back through model.py's pass, part by part: its size (all its numbers measured together) at the
    scores, at the block's top, into each add's branch, below each add, and at the rows the tables made."""
    blk = model.blocks[0]
    rows = model.tok_emb(x) + model.pos_emb(torch.arange(x.size(1)))   # [B×T×28]  token row + position row
    att = blk.attn(blk.ln1(rows))                                 # the attention's branch
    mid = rows + att                                              # its add
    ff = blk.ff(blk.ln2(mid))                                     # the feed-forward's branch
    top = mid + ff                                                # its add: the block's top
    scores = model.ln_f(top) @ model.tok_emb.weight.T + model.head_bias   # [B×T×65]  the readout
    loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
    parts = {"scores": scores, "top": top, "ff_branch": ff, "meet": mid, "attn_branch": att, "rows": rows}
    grads = torch.autograd.grad(loss, list(parts.values()))       # every part's gradient, in one pass back
    return {name: g.norm().item() for name, g in zip(parts, grads)}, dict(zip(parts, grads))


def table_gradient_by_use(model, x, y):
    """The token table's gradient, one job at a time: read in at the start (each input's entry), and reused at the readout (scoring
    every character). Holding one use still (detach) leaves the other's push; the two add up to the whole gradient."""
    W = model.tok_emb.weight                                      # [65×28]

    def push(use):
        rows = (W if use == "in" else W.detach())[x] + model.pos_emb(torch.arange(x.size(1)))
        for b in model.blocks:
            rows = b(rows)
        scores = model.ln_f(rows) @ (W if use == "out" else W.detach()).T + model.head_bias
        loss = torch.nn.functional.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
        return torch.autograd.grad(loss, W)[0]                    # [65×28]

    return push("in"), push("out")


# ---------------------------------------------------------------------------------------------------- T4 · AdamW

def adamw_step(p, g, m, v, t, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW step for any tensor of numbers, as torch.optim.AdamW takes it on PyTorch's defaults (train.py passes only the
    learning rate). m and v are the running averages of the pushes and of their squares, both 0 before step 1; t counts the
    steps from 1. Returns the moved numbers and the two averages."""
    b1, b2 = betas
    p = p * (1 - lr * weight_decay)                               # the W: weight decay, a pull toward 0 (T4.4)
    m = b1 * m + (1 - b1) * g                                     # the pushes, averaged with their signs (T4.3)
    v = b2 * v + (1 - b2) * g * g                                 # their sizes, squared and averaged
    m_hat = m / (1 - b1 ** t)                                     # both start at 0: the corrections (T4.2)
    v_hat = v / (1 - b2 ** t)
    step = m_hat / (v_hat.sqrt() + eps)                           # each push over its own size: about ±1 at step 1 (T4.1)
    return p - lr * step, m, v


def steps_in_rates(pushes, betas=(0.9, 0.999), eps=1e-8, corrected=True):
    """One number's move at each step, in learning rates (weight decay aside), for a list of its pushes."""
    b1, b2 = betas
    m = v = 0.0
    moves = []
    for t, g in enumerate(pushes, 1):
        m, v = b1 * m + (1 - b1) * g, b2 * v + (1 - b2) * g * g
        if corrected:
            moves.append(-(m / (1 - b1 ** t)) / (math.sqrt(v / (1 - b2 ** t)) + eps))
        else:
            moves.append(-m / (math.sqrt(v) + eps))               # without the corrections
    return moves


def decay_left(rates, weight_decay=0.01):
    """What weight decay alone leaves of a number: each step keeps 1 − weight_decay × that step's learning rate."""
    left = 1.0
    for lr in rates:
        left *= 1 - weight_decay * lr
    return left


def learning_rate(step, lr=3e-3, min_lr=3e-4, warmup=200, max_steps=100_000):
    """train.py's schedule (lr_at): a straight climb to lr over the warm-up, then down a cosine to min_lr at the last step."""
    if step < warmup:
        return lr * step / warmup
    done = min(1.0, (step - warmup) / (max_steps - warmup))
    return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * done))


def first_steps(model, train_ids, steps, rate=learning_rate, B=64, T=32, watch=None):
    """The run's first steps, as train.py takes them (the seed set just before the model was made): each step's learning rate,
    64 random windows, their loss, the way back, then AdamW's step. Calls watch(model) after every step; returns its answers."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    seen = []
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = rate(step + 1)                          # this step's learning rate
        starts = torch.randint(len(train_ids) - T - 1, (B,))      # this step's 64 places
        x = torch.stack([train_ids[i:i + T] for i in starts])
        y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
        loss = model(x, y)[1]
        opt.zero_grad(set_to_none=True)
        loss.backward()                                           # the way back
        opt.step()                                                # AdamW moves every number
        if watch:
            seen.append(watch(model))
    return seen


@torch.no_grad()
def check_loss(model, ids, starts, T=32):
    """The run's check: the average loss over fixed windows (check_starts), as train.py scores it every 500 steps."""
    model.eval()
    total = sum(model(torch.stack([ids[i:i + T] for i in s]), torch.stack([ids[i + 1:i + T + 1] for i in s]))[1].item() for s in starts)
    model.train()
    return total / len(starts)


# ---------------------------------------------------------------------------------------------------- T5 · the loop

def one_turn(model, opt, train_ids, step, B=64, T=32, clear=True):
    """One turn of train.py's loop, in its order: this step's learning rate, 64 random windows, their loss, the old gradients
    cleared, the way back, then AdamW's step. clear=False leaves the old gradients in (T5.1). Returns the turn's loss."""
    for group in opt.param_groups:
        group["lr"] = learning_rate(step + 1)                     # 1 · this step's learning rate
    starts = torch.randint(len(train_ids) - T - 1, (B,))          # 2 · 64 random places
    x = torch.stack([train_ids[i:i + T] for i in starts])
    y = torch.stack([train_ids[i + 1:i + T + 1] for i in starts])
    loss = model(x, y)[1]                                         # 3 · their loss
    if clear:
        opt.zero_grad(set_to_none=True)                           # 4 · the last turn's gradients cleared
    loss.backward()                                               # 5 · the way back adds each gradient to what .grad holds
    opt.step()                                                    # 6 · AdamW moves every number
    return loss.item()


def run_turns(model, train_ids, steps, check=None, every=50, clear=True, remake=False):
    """Turns 1 to `steps` from where the model is, scoring check(model) every `every` steps. remake=True makes a new AdamW every
    turn, so its averages and its step count are lost (T5.2). Returns [[step, check], ...]."""
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    curve = []
    for step in range(steps + 1):
        if check and step % every == 0:
            curve.append([step, check(model)])
        if step == steps:
            return curve
        if remake:
            opt = torch.optim.AdamW(model.parameters(), lr=3e-3)  # a new AdamW: its averages start at 0 again
        one_turn(model, opt, train_ids, step, clear=clear)


def what_lasts(model, opt):
    """What carries on from one turn to the next: the learned numbers, AdamW's two averages for each, and its step count."""
    numbers = sum(p.numel() for p in model.parameters())
    averages = sum(s["exp_avg"].numel() + s["exp_avg_sq"].numel() for s in opt.state.values())
    steps = sorted({int(s["step"]) for s in opt.state.values()})
    return numbers, averages, steps


def saved(path):
    """What a saved file holds: each table's name and shape, how many numbers in all, and the file's size in bytes."""
    tables = torch.load(path, weights_only=True)
    shapes = {name: tuple(t.shape) for name, t in tables.items()}
    return shapes, sum(t.numel() for t in tables.values()), os.path.getsize(path)

In [6]:
tok = Tokenizer(text)
cfg = Config(vocab_size=len(tok.chars))
ids = torch.tensor(tok.encode(text))                        # the whole text as ids
cut = int(0.9 * len(ids))
train_ids = ids[:cut]                                       # the first 90% trains

# step 0, in train.py's order: the seed, then the model (its starting numbers), then the first batch's 64 starts
torch.manual_seed(1337)
model0 = Prompter(cfg)
starts = torch.randint(len(train_ids) - 32 - 1, (64,))
x0 = torch.stack([train_ids[i:i + 32] for i in starts])        # [64×32]  the inputs
y0 = torch.stack([train_ids[i + 1:i + 33] for i in starts])    # [64×32]  their right answers
assert starts.tolist() == [160204, 793724, 810574, 964722, 40734, 423895, 818722, 483307, 510490, 928356, 614059, 30062, 737496, 407796, 59365, 998494, 864171, 929951, 942896, 637372, 408276, 289366, 811280, 165163, 748492, 428999, 260643, 560439, 371836, 390719, 776246, 678592, 338694, 316420, 711307, 461584, 554475, 251253, 546994, 358001, 64331, 976218, 589122, 206049, 243936, 44076, 936280, 819204, 162295, 566156, 935951, 895342, 457178, 537830, 274346, 823229, 14080, 899619, 528867, 495983, 90293, 96640, 21128, 975278]   # the recorded run's first batch
space, dollar = tok.chars.index(' '), tok.chars.index('$')
with torch.no_grad():
    loss0 = model0(x0, y0)[1].item()
print(f"the first batch: {tuple(x0.shape)} · its loss at step 0: {loss0:.4f} · {count_params(model0):,} learned numbers")
assert abs(loss0 - 4.17812) < 1e-4 and count_params(model0) == 9417

the first batch: (64, 32) · its loss at step 0: 4.1781 · 9,417 learned numbers


# Section 1 · One number's gradient

One learned number's gradient, found two ways. Each chapter below follows its page: Picture it, Watch it in our run, then Read and
run the code.

## T3.1 · The readout's biases

*What decides a readout bias's gradient?*

**Predict first** (the page asks it too, before its clip): at step 0 the model gives every character about 1 in 65. In the first batch, which character's readout bias gets the strongest
push up, the most negative gradient: the space, the most common right answer; $, the rarest character, as it needs the most help;
or none stands out, as every guess is about even?

### T3.1.1 · Picture it

One learned number is used by every answer of a batch. What decides its gradient? Picture one made-up number, used by 6 made-up
answers. Each answer adds its own part: negative if raising the number would lower that answer's cost (−0.6, 0.1, 0.2, −0.3, 0.1
and −0.1). The gradient is their parts, averaged: −0.1. Negative means raising the number lowers the loss: a push up. (The page's
parts are made up to show the idea.)

### T3.1.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here are the first batch's 65 readout biases at step 0, each with its gradient. The space is the right answer 329
times in 2,048: a share of 0.16. Yet it was given only 0.016. Its gradient is 0.016
minus 0.16, which is −0.144: the strongest push up. $ is never right, so its gradient is just its
probability, 0.017: a push down. Each bar is its character's average probability, minus its share of the right
answers.

### T3.1.3 · Read and run the code

`bias_gradient` in `code/training_steps.py` (copied in above) works out every bias's gradient by the rule: each character's
probability averaged over the places, minus its share of the right answers. The cell runs backward on the first batch:
`head_bias.grad` agrees with the rule to 7 decimal places. So for a readout bias, backpropagation's whole answer is this one rule.

In [7]:
model0.zero_grad()
model0(x0, y0)[1].backward()                                # backpropagation: every learned number's .grad
with torch.no_grad():
    rule = bias_gradient(torch.softmax(model0(x0)[0], dim=-1), y0)   # [65]  average probability − share of right answers
auto = model0.head_bias.grad
diff = (rule - auto).abs().max().item()
print(f"the rule against backpropagation, the largest difference: {diff:.10f} (only rounding)")
print(f"the space: {rule[space].item():.4f} · $: {rule[dollar].item():.4f}")
print(f"all 65 gradients added up: {rule.sum().item():.7f}: the pushes up and the pushes down balance out")
print('pushed up most:', [(tok.chars[i], round(rule[i].item(), 3)) for i in rule.argsort()[:3].tolist()])
assert (rule - auto).abs().max().item() < 1e-7
assert abs(rule[space].item() - -0.144349) < 1e-6 and abs(rule[dollar].item() - 0.0166319) < 1e-6
assert int(rule.argmin()) == space

the rule against backpropagation, the largest difference: 0.0000000075 (only rounding)
the space: -0.1443 · $: 0.0166
all 65 gradients added up: -0.0000000: the pushes up and the pushes down balance out
pushed up most: [(' ', -0.144), ('e', -0.074), ('t', -0.05)]


### ✋ Your turn

Which character's bias gets the strongest push down, the largest gradient? Find it in `rule` with `argmax`, and put the character
(a string, from `tok.chars`) in `answer`.

In [8]:
answer = None   # TODO: tok.chars[int(rule.argmax())]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[int(rule.argmax())], 'the index of the largest gradient: int(rule.argmax()); its character: tok.chars[...]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [9]:
#@title Show a solution { display-mode: "form" }
answer = tok.chars[int(rule.argmax())]
print(repr(answer), 'is never a right answer here:', int((y0 == tok.chars.index(answer)).sum()), 'times')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[int(rule.argmax())], 'the index of the largest gradient: int(rule.argmax()); its character: tok.chars[...]'
    print("✓ right:", answer)

'$' is never a right answer here: 0 times
✓ right: $


## T3.2 · Check it by nudging

*How can you check a gradient without backpropagation?*

**Predict first** (the page asks it too, before its clip): to check every gradient by nudging, as the picture did, how many runs of the batch would all 9,417 learned numbers take:
18,834, an up and a down for each; 2, all of them up at once, then all down; or 1, as one pass gives them all?

### T3.2.1 · Picture it

Backpropagation hands every learned number a gradient. How could you check one? Picture a made-up model with 3 numbers. Its loss is
2.30. Nudge only the first up by 0.01, and run it: the loss reads 2.32. Down by 0.01: 2.28. The loss moved 0.04 across 0.02: a
slope of 2. That is the first number's gradient. (The page's model is made up to show the idea.)

### T3.2.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is the first batch's loss at step 0, as the space's bias is nudged. Nudged up by 0.001, the loss falls by
0.000144. Nudged down, it rises by 0.000144. The slope through the two runs is −0.1445.
Backpropagation said −0.1443: the first 3 decimal places match. But that took 2 runs for one number. All 9,417
would take 18,834; backpropagation finds every one in one pass back.

### T3.2.3 · Read and run the code

`nudge` moves one number up and runs the batch, then down, and runs it again. It returns the slope through the two losses, and puts
the number back as it was. Then `grad_twice` runs backward: every number's `.grad` is filled in one pass, and none moves. The
space's bias itself is still 0. Run backward again without clearing, and the gradients add up. So `train.py` clears them first, with
`opt.zero_grad()`.

In [10]:
slope = nudge(model0, x0, y0, model0.head_bias, space, h=1e-3)
once, again = grad_twice(model0, x0, y0, model0.head_bias)
print(f"nudged ±0.001: {slope:.4f} · backpropagation: {once[space].item():.4f} · the bias itself: {model0.head_bias[space].item()}")
print(f"backward again, not cleared: {again[space].item():.4f}, twice the first")
print(f"to check all {count_params(model0):,} numbers by nudging: {2 * count_params(model0):,} runs of the batch")
assert abs(slope - -0.144482) < 1e-6 and round(slope, 3) == round(once[space].item(), 3)
assert model0.head_bias[space].item() == 0.0 and (again - 2 * once).abs().max().item() < 1e-7
model0.zero_grad()                                          # cleared, as train.py does before every backward

nudged ±0.001: -0.1445 · backpropagation: -0.1443 · the bias itself: 0.0
backward again, not cleared: -0.2887, twice the first
to check all 9,417 numbers by nudging: 18,834 runs of the batch


### ✋ Your turn

Why nudge by 0.001, and not by something tiny? Nudge the space's bias by `h=1e-7` instead, and put the slope in `answer`. The
model's numbers are 32-bit: a loss holds only about 7 digits.

In [11]:
answer = None   # TODO: nudge(model0, x0, y0, model0.head_bias, space, h=1e-7)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == nudge(model0, x0, y0, model0.head_bias, space, h=1e-7), 'the same call with h=1e-7: nudge(model0, x0, y0, model0.head_bias, space, h=1e-7)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [12]:
#@title Show a solution { display-mode: "form" }
answer = nudge(model0, x0, y0, model0.head_bias, space, h=1e-7)
print('nudged by 0.0000001, the slope reads', answer, ': too small a change for a 32-bit loss to hold')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == nudge(model0, x0, y0, model0.head_bias, space, h=1e-7), 'the same call with h=1e-7: nudge(model0, x0, y0, model0.head_bias, space, h=1e-7)'
    print("✓ right:", answer)

nudged by 0.0000001, the slope reads 0.0 : too small a change for a 32-bit loss to hold
✓ right: 0.0


# Section 2 · Backward through the block

The gradient followed back from the loss, through the block's adds, to the tables.

## T3.3 · Through the adds

*How does the gradient travel back through the block?*

**Predict first** (the page asks it too, before its clip): our block has two adds: x plus attention, then plus the feed-forward. At an add, the gradient arriving from above goes: whole, down
both paths, the branch and the straight path; half down each path, as the two share it; or only into the branch, as the straight
path has nothing to learn?

### T3.3.1 · Picture it

How does backpropagation carry the gradient back from the loss, one part at a time? Picture a made-up chain: a number, times 3, then the
loss. The loss hands back its slope, 0.5. Each part multiplies what arrives by its own slope: times 3 makes 1.5. So each part needs
only its own slope; multiplied along the chain, they give the gradient. That is the chain rule. (The page's chain is made up to
show the idea.)

### T3.3.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is the first batch's gradient, followed back into the block. At the block's top, its size (all its numbers, measured
together) is 0.079. At the feed-forward's add it goes whole, both ways: 0.079 into the branch, and
0.079 straight down. The branch hands back less, through its own slopes. Where the paths meet, the two add, number by
number: 0.082. (Sizes do not add: the two paths' numbers do.) The attention's add hands that on whole, both ways. Back
at the rows the tables made, it is 0.083: the straight path carried it down, no smaller.

### T3.3.3 · Read and run the code

`backward_sizes` runs `model.py`'s pass part by part, to find each part's gradient. Its two adds are the block's own lines: rows
plus attention, then plus the feed-forward. One call to `torch.autograd.grad` brings every part's gradient back, through each linear
layer and LayerNorm, in one pass. The cell checks that each add hands both paths the very same numbers.

In [13]:
sizes, grads = backward_sizes(model0, x0, y0)
for name, v in sizes.items():
    print(f"{name:>12}: {v:.3f}")
print('the feed-forward add: the same numbers both ways?', torch.equal(grads['ff_branch'], grads['top']))
print('the attention add:    the same numbers both ways?', torch.equal(grads['attn_branch'], grads['meet']))
assert torch.equal(grads['ff_branch'], grads['top']) and torch.equal(grads['attn_branch'], grads['meet'])
for k, v in (('top', 0.0785918), ('meet', 0.0815319), ('rows', 0.0827638)):
    assert abs(sizes[k] - v) < 1e-6                         # the run's sizes
assert sizes['rows'] >= sizes['top']                        # no smaller at the rows the tables made

      scores: 0.022
         top: 0.079
   ff_branch: 0.079
        meet: 0.082
 attn_branch: 0.082
        rows: 0.083
the feed-forward add: the same numbers both ways? True
the attention add:    the same numbers both ways? True


### ✋ Your turn

One part's gradient is exactly 0 in exact arithmetic: the 28 key biases. A key bias adds the same amount to every
score a question gives its labels, so it changes no share. Run backward, take the key biases' gradient (the middle 28 of the
attention's `qkv.bias.grad`), and put its largest size in `answer`. Is it under 0.000000001?

In [14]:
model0.zero_grad()
model0(x0, y0)[1].backward()
kb = model0.blocks[0].attn.qkv.bias.grad[28:56]   # the 28 key biases
answer = None   # TODO: kb.abs().max().item()

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - kb.abs().max().item()) < 1e-15 and answer < 1e-9, 'the largest size: kb.abs().max().item()'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [15]:
#@title Show a solution { display-mode: "form" }
model0.zero_grad()
model0(x0, y0)[1].backward()
kb = model0.blocks[0].attn.qkv.bias.grad[28:56]
answer = kb.abs().max().item()
print(f"the key biases' largest gradient: {answer:.13f}, only rounding")
model0.zero_grad()

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - kb.abs().max().item()) < 1e-15 and answer < 1e-9, 'the largest size: kb.abs().max().item()'
    print("✓ right:", answer)

the key biases' largest gradient: 0.0000000000012, only rounding
✓ right: 1.170619157164765e-12


## T3.4 · One table, two jobs

*Where does the token table's gradient come from?*

**Predict first** (the page asks it too, before its clip): $ is never read in by the first batch, and never its right answer. Does its entry in the token table get a push: yes, as the readout
scores $ with that entry; no, as an entry is pushed only where it is read in; or only its readout bias does, as the entry is for
reading in?

### T3.4.1 · Picture it

The token table gives every character an entry of numbers. Which entries does a batch push? Picture a made-up table of 3 characters, and
inputs that read in only a and b. From the start of the pass, pushes come back to the entries that were read in: the entries for a
and b. c is never read in, so from the start, its entry gets nothing. (The page's table is made up to show the idea.)

### T3.4.2 · Watch it in our run

*(The page's recorded clip, in words.)*

Here is the token table, a matrix of 65 entries, at both its jobs: reading in each input, and scoring every character. The first
batch reads in 56 of the 65 characters: those entries get a push from the start. The readout scores all 65, so every
entry gets a push from there. $, never read in, gets 0.023, all from the readout. The space's entry gets
0.128 from being read in, 0.201 from the readout. Added number by number, they make one push of
0.288. (Not 0.128 + 0.201: sizes do not add, the numbers do.)

### T3.4.3 · Read and run the code

`table_gradient_by_use` finds the token table's gradient one job at a time. It holds the table still at one job with `detach`, so
only the other job pushes; then it swaps. The two pushes add up to the table's whole gradient: the cell checks it, entry by entry.
PyTorch does this for any number used twice: it adds the pushes from every use.

In [16]:
g_in, g_out = table_gradient_by_use(model0, x0, y0)
model0.zero_grad()
model0(x0, y0)[1].backward()
whole = model0.tok_emb.weight.grad
print(f"the two jobs added, against the whole gradient, the largest difference: {(g_in + g_out - whole).abs().max().item():.7f}")
print('entries pushed from the start:', int((g_in.abs().sum(1) > 0).sum()), ' from the readout:', int((g_out.abs().sum(1) > 0).sum()))
print(f"$: {g_in[dollar].norm().item():.3f} from the start, {g_out[dollar].norm().item():.3f} from the readout")
print(f"the space: {g_in[space].norm().item():.3f} from the start, {g_out[space].norm().item():.3f} from the readout, {(g_in[space] + g_out[space]).norm().item():.3f} added")
assert (g_in + g_out - whole).abs().max().item() < 1e-7
assert int((g_in.abs().sum(1) > 0).sum()) == 56 and int((g_out.abs().sum(1) > 0).sum()) == 65
assert g_in[dollar].abs().sum().item() == 0 and abs(g_out[dollar].norm().item() - 0.0229407) < 1e-6
assert abs((g_in[space] + g_out[space]).norm().item() - 0.287745) < 1e-6   # added number by number
model0.zero_grad()

the two jobs added, against the whole gradient, the largest difference: 0.0000000


entries pushed from the start: 56  from the readout: 65
$: 0.000 from the start, 0.023 from the readout
the space: 0.128 from the start, 0.201 from the readout, 0.288 added


### ✋ Your turn

How many of the 65 characters does the first batch read in? Count the different ids in `x0` (a set of `x0.flatten().tolist()`), and
put the number in `answer`. It should match the entries pushed from the start.

In [17]:
answer = None   # TODO: len(set(x0.flatten().tolist()))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == len(set(x0.flatten().tolist())), 'the different ids: set(x0.flatten().tolist()); how many: len(...)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [18]:
#@title Show a solution { display-mode: "form" }
answer = len(set(x0.flatten().tolist()))
print(answer, 'characters read in: the entries pushed from the start')

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == len(set(x0.flatten().tolist())), 'the different ids: set(x0.flatten().tolist()); how many: len(...)'
    print("✓ right:", answer)

56 characters read in: the entries pushed from the start
✓ right: 56


One pass back gives every learned number its gradient. For a readout bias, it is the bias's average probability minus its share of
the right answers; a nudge checks it, but would need 18,834 runs for all 9,417. On the way back each part multiplies
what arrives by its own slope, and an add hands it whole to both paths, so it reaches the rows the tables made no smaller. The
token table, used twice, adds both pushes, number by number: even $, never read in, gets one from the readout.

Next: **T4 · AdamW, step by step**, which turns those gradients into steps.